# Step 5: Ablation study

Each variant removes or isolates one group of candidate features and is **retrained from scratch**. The user–article matching group U stays in every variant, so changes come from candidate-side evidence only.

| Variant | T1 behaviour | T2 transferred | T3 content | U matching | H content_smoothed_ctr |
|---|:-:|:-:|:-:|:-:|:-:|
| `full` | ✓ | ✓ | ✓ | ✓ | |
| `no_t1` | | ✓ | ✓ | ✓ | |
| `no_t2` | ✓ | | ✓ | ✓ | |
| `no_t3` | ✓ | ✓ | | ✓ | |
| `t1_only` | ✓ | | | ✓ | |
| `t3_only` | | | ✓ | ✓ | |
| `full_plus_smoothing` | ✓ | ✓ | ✓ | ✓ | ✓ |

**Protocol.** Every variant uses the LambdaRank objective and the tree shape selected in step 4 and its own early stopping on validation. For `full_plus_smoothing`, α is chosen on validation from {5, 10, 20, 50, 100} with the shared selection rule. Test is scored once per variant, after all choices are fixed.

"Without Tier 3" removes the explicit content columns only. Content still reaches the model through Tier 2 (transferred behaviour) and U (similarities), so it measures the value of the explicit content inputs given everything else. Contributions need not add up, because groups overlap.

> Metrics here come from `shared/eval_interim.py`. Final ablation tables, segment analysis and uncertainty come from `step6_evaluation_diagnostics.ipynb`.

## 1. Setup

In [1]:
import os, sys, subprocess, json, time

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.getcwd()
    DATA_ROOT = os.environ.get("G11_DATA_ROOT", os.path.join(REPO_DIR, "data"))
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
from shared import schema as S, feature_groups as FG, lgbm_pipeline as L, selection as SEL
from shared.predictions import write_predictions
from shared import eval_interim as E      # swap for the team evaluator once it exists

DATA_KIND = "REAL"   # recorded in every prediction file
import kagglehub
RAW_DIR = os.path.join(kagglehub.dataset_download("arashnic/mind-news-dataset"), "MINDsmall_train")
FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
version = S.read_version(FEATURE_ROOT)
assert version.get("data_kind", "REAL") == DATA_KIND, f"feature tables are {version.get('data_kind')}, run is {DATA_KIND}"
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")
MODEL_DIR = os.path.join(DATA_ROOT, "models", DATA_KIND.lower())
os.makedirs(MODEL_DIR, exist_ok=True)
SEED = 4222
N_TITLE_PC, N_ENT_PC = 32, 16
INCLUDE_USER_PC = True      # user profile vectors projected with the article PCA; set False if RAM runs short
SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10", "cold_hit@10", "warm_recall@10"]
print(f"DATA_KIND={DATA_KIND} | features: {FEATURE_ROOT} | schema {version['schema_version']}")

with open(os.path.join(MODEL_DIR, "lgbm_selected_params.json")) as f:
    chosen = json.load(f)
assert (chosen["n_title_pc"], chosen["n_ent_pc"], chosen["include_user_pc"]) == (N_TITLE_PC, N_ENT_PC, INCLUDE_USER_PC)
assert chosen.get("objective", "binary") == L.BASE_PARAMS["objective"], "re-run step 4 first"
PARAMS = chosen["params"]
print("tree shape from step 4:", PARAMS)

C:\Users\Bryan Ng\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DATA_KIND=REAL | features: C:\Users\Bryan Ng\g11_data\features\v0.1 | schema 0.1
tree shape from step 4: {'num_leaves': 31, 'min_data_in_leaf': 500}


## 2. Reduce vectors and load the row tables (same as step 4)

In [2]:
t0 = time.time()
train_meta_ids = pd.read_parquet(S.meta_path(FEATURE_ROOT, "train"), columns=["impression_id", "news_id"])
accessible = set(train_meta_ids["news_id"].astype(str)) | L.train_histories(RAW_DIR, train_meta_ids["impression_id"].unique())
reducer = L.VectorReducer(N_TITLE_PC, N_ENT_PC, seed=SEED).fit(FEATURE_ROOT, accessible)
print(f"PCA fitted on {reducer.n_fit_articles:,} training-accessible articles "
      f"({reducer.n_fit_entity_articles:,} with entity vectors); explained variance {reducer.explained_variance()}")

data = {split: L.load_split(FEATURE_ROOT, split, reducer, include_user_pc=INCLUDE_USER_PC) for split in S.SPLITS}
INCLUDE_SEMANTIC = "semantic_neighbor_ctr" in data["train"][1].columns   # optional in the contract
GROUPS = FG.group_columns(N_TITLE_PC, N_ENT_PC, include_user_pc=INCLUDE_USER_PC, include_semantic=INCLUDE_SEMANTIC)
print("semantic_neighbor_ctr present:", INCLUDE_SEMANTIC)
pd.DataFrame({split: {"rows": len(m), "impressions": m["impression_id"].nunique(), "columns available": x.shape[1],
                      "memory MB": round(x.memory_usage(deep=True).sum() / 1e6, 1)}
              for split, (m, x) in data.items()}).T

PCA fitted on 43,370 training-accessible articles (37,368 with entity vectors); explained variance {'title': 0.4252181649208069, 'entity': 0.7117447853088379}


semantic_neighbor_ctr present: True


,rows,impressions,columns available,memory MB
train,3389342.0,95071.0,131.0,1776.0
val,1231673.0,31624.0,131.0,645.4
test,1222429.0,30270.0,131.0,640.6


## 3. Choose α for `full_plus_smoothing` on validation

In [3]:
(m_tr, x_tr), (m_va, x_va) = data["train"], data["val"]
g_tr, g_va = L.impression_groups(m_tr), L.impression_groups(m_va)   # LambdaRank ranks within impressions
cols_h = FG.feature_columns("full_plus_smoothing", n_title_pc=N_TITLE_PC, n_ent_pc=N_ENT_PC, include_user_pc=INCLUDE_USER_PC,
                         include_semantic=INCLUDE_SEMANTIC)
alpha_rows, alpha_models = [], {}
for a in S.ALPHA_GRID:
    b = L.fit(L.design(x_tr, cols_h, a), m_tr["label"], L.design(x_va, cols_h, a), m_va["label"], PARAMS, seed=SEED,
              group_tr=g_tr, group_va=g_va)
    alpha_models[a] = b
    res = E.evaluate(m_va, L.predict(b, L.design(x_va, cols_h, a)))
    alpha_rows.append({"alpha": a, "best_iteration": b.best_iteration, **{k: res[k] for k in SHOW}})
alpha_table = pd.DataFrame(alpha_rows).set_index("alpha")
ALPHA = int(SEL.select(alpha_table))
print(f"selected alpha = {ALPHA}: {SEL.describe()}")
alpha_table.round(4)

selected alpha = 10: highest validation cold_recall@10 among settings whose validation ndcg@10 is within 0.005 of the best; ties broken on ndcg@10


,best_iteration,auc,mrr,ndcg@5,ndcg@10,cold_recall@10,cold_hit@10,warm_recall@10
alpha,,,,,,,,
5,274,0.7164,0.3621,0.3955,0.4529,0.6709,0.6823,0.6877
10,387,0.7186,0.3635,0.3973,0.4542,0.6720,0.6855,0.6884
20,410,0.7184,0.3630,0.3966,0.4537,0.6542,0.6663,0.6881
50,254,0.7165,0.3615,0.3943,0.4516,0.6397,0.6535,0.6859
100,333,0.7177,0.3625,0.3961,0.4535,0.6554,0.6674,0.6888


**Takeaway: α.** nDCG@10 barely moves across α (0.452 to 0.454), while Cold Recall@10 ranges from 0.64 to 0.67. α = 10 gives the best value of both, so it is selected.

## 4. Train every variant and score all splits

In [4]:
per_imp, rows = {}, []
for variant in FG.ABLATIONS:
    cols = FG.feature_columns(variant, n_title_pc=N_TITLE_PC, n_ent_pc=N_ENT_PC, include_user_pc=INCLUDE_USER_PC,
                         include_semantic=INCLUDE_SEMANTIC)
    a = ALPHA if "content_smoothed_ctr" in cols else None
    t0 = time.time()
    b = alpha_models[ALPHA] if variant == "full_plus_smoothing" else \
        L.fit(L.design(x_tr, cols, a), m_tr["label"], L.design(x_va, cols, a), m_va["label"], PARAMS, seed=SEED,
              group_tr=g_tr, group_va=g_va)
    cfg = L.param_id(PARAMS) + (f",alpha={a}" if a else "")
    for split, (m, x) in data.items():
        sc = L.predict(b, L.design(x, cols, a))
        write_predictions(m["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id=f"lgbm_{variant}", config_id=cfg,
                          split=split, schema_version=S.SCHEMA_VERSION,
                          extra={"variant": variant, "n_columns": len(cols), "best_iteration": b.best_iteration, "selected_on": "val"})
        pi = E.per_impression_metrics(m, sc)
        per_imp[(variant, split)] = pi
        rows.append({"variant": variant, "split": split, "n_columns": len(cols), "best_iteration": b.best_iteration,
                     **{k: v for k, v in E.summarise(pi).items() if k in SHOW}})
    print(f"{variant:20s} {len(cols):3d} columns, {b.best_iteration} trees, {time.time() - t0:.0f}s")
ablation = pd.DataFrame(rows).set_index(["split", "variant"])
ablation.loc["val"].round(4)

full                 122 columns, 432 trees, 106s


no_t1                114 columns, 390 trees, 95s


no_t2                114 columns, 398 trees, 95s


no_t3                 70 columns, 336 trees, 62s


t1_only               62 columns, 222 trees, 49s


t3_only              106 columns, 508 trees, 102s


full_plus_smoothing  123 columns, 387 trees, 23s


,n_columns,best_iteration,mrr,ndcg@5,ndcg@10,auc,cold_recall@10,cold_hit@10,warm_recall@10
variant,,,,,,,,,
full,122,432,0.3630,0.3968,0.4533,0.7166,0.6507,0.6631,0.6870
no_t1,114,390,0.3394,0.3668,0.4259,0.6863,0.5503,0.5650,0.6624
no_t2,114,398,0.3627,0.3961,0.4533,0.7168,0.6508,0.6631,0.6877
no_t3,70,336,0.3556,0.3874,0.4452,0.7104,0.6537,0.6663,0.6801
t1_only,62,222,0.3532,0.3839,0.4422,0.7070,0.6707,0.6834,0.6762
t3_only,106,508,0.3391,0.3670,0.4260,0.6861,0.5473,0.5597,0.6640
full_plus_smoothing,123,387,0.3635,0.3973,0.4542,0.7186,0.6720,0.6855,0.6884


In [5]:
ablation.loc["test"].round(4)

,n_columns,best_iteration,mrr,ndcg@5,ndcg@10,auc,cold_recall@10,cold_hit@10,warm_recall@10
variant,,,,,,,,,
full,122,432,0.3376,0.3695,0.4306,0.7141,0.6054,0.6237,0.6727
no_t1,114,390,0.3048,0.3297,0.3914,0.6676,0.5111,0.5305,0.6325
no_t2,114,398,0.3356,0.3674,0.4286,0.7128,0.5889,0.6081,0.6714
no_t3,70,336,0.3326,0.3613,0.4228,0.7056,0.5981,0.6189,0.6613
t1_only,62,222,0.3321,0.3604,0.4223,0.7041,0.6037,0.6237,0.6612
t3_only,106,508,0.3040,0.3294,0.3899,0.6642,0.5130,0.5305,0.6297
full_plus_smoothing,123,387,0.3369,0.3691,0.4296,0.7137,0.6247,0.6428,0.6709


## 5. Change against the full model

Each variant minus `full` on the same impressions. For Cold Recall@10 and nDCG@10 the 95% interval is a paired bootstrap over impressions (1,000 resamples). Read validation for decisions; test confirms them.

In [6]:
delta_rows = []
for split in ("val", "test"):
    base = per_imp[("full", split)]
    for variant in FG.ABLATIONS:
        if variant == "full":
            continue
        other = per_imp[(variant, split)]
        row = {"split": split, "variant": variant}
        for metric in ("ndcg@10", "cold_recall@10"):
            bs = E.paired_bootstrap(base[metric], other[metric])
            row.update({f"Δ {metric}": bs["mean_diff"], f"{metric} CI": f"[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]"})
        row["cold-click impressions"] = int(base["cold_recall@10"].notna().sum())
        delta_rows.append(row)
pd.DataFrame(delta_rows).set_index(["split", "variant"]).round(4)

Δ ndcg@10          ndcg@10 CI  Δ cold_recall@10  \
split variant                                                                
val   no_t1                  -0.0274  [-0.0295, -0.0252]           -0.1004   
      no_t2                  -0.0000  [-0.0012, +0.0011]            0.0001   
      no_t3                  -0.0081  [-0.0102, -0.0060]            0.0030   
      t1_only                -0.0111  [-0.0131, -0.0090]            0.0200   
      t3_only                -0.0273  [-0.0294, -0.0251]           -0.1034   
      full_plus_smoothing     0.0009  [-0.0004, +0.0021]            0.0213   
test  no_t1                  -0.0392  [-0.0416, -0.0369]           -0.0944   
      no_t2                  -0.0021  [-0.0034, -0.0008]           -0.0165   
      no_t3                  -0.0078  [-0.0099, -0.0058]           -0.0074   
      t1_only                -0.0083  [-0.0103, -0.0062]           -0.0017   
      t3_only                -0.0407  [-0.0431, -0.0383]           -0.0924   
      full_plus_smoothing    -0.0011  [-0.0024, +0.0002]            0.0192   

                            cold_recall@10 CI  cold-click impressions  
split variant                                                          
val   no_t1                [-0.1246, -0.0749]                     938  
      no_t2                [-0.0179, +0.0201]                     938  
      no_t3                [-0.0197, +0.0284]                     938  
      t1_only              [-0.0027, +0.0448]                     938  
      t3_only              [-0.1279, -0.0776]                     938  
      full_plus_smoothing  [+0.0030, +0.0419]                     938  
test  no_t1                [-0.1196, -0.0686]                     837  
      no_t2                [-0.0354, +0.0024]                     837  
      no_t3                [-0.0311, +0.0157]                     837  
      t1_only              [-0.0267, +0.0218]                     837  
      t3_only              [-0.1173, -0.0671]                     837  
      full_plus_smoothing  [+0.0002, +0.0385]                     837

## 6. What the ablations show

* **T1, the article's own behaviour, is essential, and for cold articles too.** Removing it costs 0.027 (validation) and 0.039 (test) nDCG@10 and about 0.10 Cold Recall@10 on both splits, all with intervals below zero. `t3_only`, which also drops T2, loses about the same. For cold articles the useful T1 signal is probably that they are new and little shown (prior exposures, article age), rather than their CTR.
* **T2, content-transferred behaviour, adds nothing measurable.** On validation, removing it changes neither metric. On test it costs 0.002 nDCG@10 (interval just below zero) and 0.017 Cold Recall@10 (interval crosses zero). The trees appear to get the same information from U and T3.
* **T3, explicit content, helps overall ranking but not cold recall specifically.** Removing it costs about 0.008 nDCG@10 on both splits (intervals below zero); the cold-recall changes are within noise. `t1_only` (behaviour plus user matching) is within 0.011 nDCG@10 of the full model and matches it on cold recall.
* **Content-smoothed CTR helps cold articles.** Adding it raises Cold Recall@10 by about 0.02 on both splits, with intervals above zero (on test only just: lower bound +0.0002), and leaves nDCG@10 unchanged. It is the only change here with a significant cold-start gain, matching the controlled comparison in step 3a.
* **For the research question:** given behaviour and user matching, content helps cold articles mainly through the content-specific smoothed prior; explicit content columns and transferred behaviour add little on top. U is kept in every variant, so this study does not measure how much the user–article match itself is worth.
* **Precision:** each split has about 840 to 940 impressions with a cold click, so Cold Recall@10 intervals are roughly ±0.02 to ±0.03, and smaller differences cannot be detected here.

*Based on the run with feature schema v0.1 and the interim evaluator. The final numbers and intervals come from step 6.*